In [1]:
import os
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"

In [2]:
!pip install wandb
# Log in once; paste the API key from https://wandb.ai/authorize

In [3]:
!pip install huggingface_hub

In [4]:
# !pip uninstall -y triton_kernels
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.4.0" "triton>=3.0.0" {install_numpy} \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 54.0 MB/s eta 0:00:00


In [5]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True

In [8]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF')
print(f"HF: {len(HF_TOKEN)}")
WB_TOKEN = userdata.get("WB")
print(f"WB: {len(WB_TOKEN)}")

HF: 37
WB: 86


In [7]:
from wandb import login
login(WB_TOKEN)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hle14838 (hle14838-h-c-vi-n-c-ng-ngh-b-u-ch-nh-vi-n-th-ng) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
from huggingface_hub import login
login(HF_TOKEN)

In [9]:
from tqdm import tqdm

In [10]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 128
dtype = None
model_name = "unsloth/gpt-oss-20b"


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    dtype = dtype, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.8.18: Fast Gpt_Oss patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


Loading weights:   0%|          | 0/3387 [00:00<?, ?it/s]

In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.
Unsloth: Detected MoE model with per-expert Linear experts. Enabling LoRA on 64 expert projection modules.


In [12]:
def formatting_prompts_func(examples, tokenizer):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

In [13]:
from datasets import load_dataset

DATASET_ID = "HuggingFaceH4/ultrachat_200k"
dataset = load_dataset(DATASET_ID)

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

data/train_sft-00000-of-00003-a3ecf92756(…): reconstructing file:   0%|          |  0.00B /  244MB            

data/train_sft-00000-of-00003-a3ecf92756(…): downloading bytes:           |  0.00B            

data/train_sft-00001-of-00003-0a1804bcb6(…): reconstructing file:   0%|          |  0.00B /  244MB            

data/train_sft-00001-of-00003-0a1804bcb6(…): downloading bytes:           |  0.00B            

...

data/test_gen-00000-of-00001-3d4cd830914(…): downloading bytes:           |  0.00B            

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

In [14]:
train_dataset = dataset["train_sft"].remove_columns(["prompt", "prompt_id"])
train_dataset = train_dataset.select(range(200)).map(formatting_prompts_func, batched=True, fn_kwargs={"tokenizer": tokenizer})
eval_dataset = dataset["test_sft"].remove_columns(["prompt", "prompt_id"])
eval_dataset = eval_dataset.select(range(100)).map(formatting_prompts_func, batched=True, fn_kwargs={"tokenizer": tokenizer})

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [15]:
# !!!!!!!!
import wave

In [16]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset, # evaluation
    args = SFTConfig(
        # Bật đánh giá
        eval_strategy="steps",   # run evaluation on the validation set...
        eval_steps=5,           # ...every 5 steps -> the eval_loss curve
        logging_steps=5,        # log train loss every 5 steps
        report_to="wandb",      # send all metrics to Weights & Biases
        dataset_text_field = "text", # Khai báo cột text chứa dữ liệu huấn luyện !!!!!!!!!!!!!!!!!!!!!!
        # After training, reload the checkpoint with the lowest eval_loss
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        max_seq_length = 2048,


        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,

        learning_rate = 2e-4,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        # packing=True, # Bật lên để TRL sẽ tự động nối các mẫu tokenized lại với nhau đến khi đạt max_seq_length.
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: You set `max_seq_length` as 2048 but the maximum the model supports is 128. We shall reduce it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/200 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/100 [00:00<?, ? examples/s]

In [17]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 199998, 'pad_token_id': 200017}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 2 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 92,454,912 of 21,007,212,096 (0.44% trained)


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Step,Training Loss,Validation Loss
5,5.536488,3.853297
10,2.762897,1.677303
15,1.324281,1.016529
20,0.966032,0.975411
25,0.982049,0.947955
30,0.836068,0.929430
35,0.866285,0.917600
40,0.808571,0.956483
45,0.753096,0.909795
50,0.807852,0.905840


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.


TrainOutput(global_step=100, training_loss=1.1342775845527648, metrics={'train_runtime': 3905.8289, 'train_samples_per_second': 0.102, 'train_steps_per_second': 0.026, 'total_flos': 6275505763123200.0, 'train_loss': 1.1342775845527648, 'epoch': 2.0})

In [22]:
model.save_pretrained("finetuned_model")

## Tiện ích

In [6]:
def download_folder(dir, downloaded_file_name):
  import shutil
  shutil.make_archive(downloaded_file_name, 'zip', dir)
  from google.colab import files
  files.download(f'{downloaded_file_name}.zip')

# Merge và export model

Nhớ khởi động lại phiên

In [12]:
!pip install "llmcompressor>=0.6.0,<=0.12.0"

  Using cached llmcompressor-0.12.0-py3-none-any.whl.metadata (13 kB)
  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.2/298.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 42.7 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully unins

In [4]:
# Tải lại để tránh xung đột khi save
!pip install --no-deps \
    "transformers==4.57.1" \
    "huggingface-hub<1.0,>=0.34.0" \
    "tokenizers" \
    "trl==0.22.2" \
    unsloth unsloth_zoo

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 544.8/544.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.5 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0


In [1]:
BASE_MODEL = "unsloth/gpt-oss-20b"
DAPTER_DIR = "finetuned_model"

In [2]:
import torch
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,   # path tới adapter đã lưu (không phải base model)
    max_seq_length = 2048,
    load_in_4bit = True,   # nên giữ True khi load để merge, unsloth tự lo phần dequant
)

==((====))==  Unsloth 2026.8.18: Fast Gpt_Oss patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gpt_oss won't work! Using float32.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# Merge LoRA vào base rồi lưu bản standalone
model.save_pretrained_merged(
    "./gpt-oss-20b-mytune-merged",
    tokenizer,
    save_method = "mxfp4",
)

Unsloth: Installing llm-compressor for FP8/FP4 export (llmcompressor>=0.6.0,<=0.12.0; pinning your torch + transformers so they are not upgraded). This can take a few minutes...
Unsloth: Merging to 16bit before MXFP4 quantization...
Unsloth: Saving full fine-tuned model to './gpt-oss-20b-mytune-merged' ...
Unsloth: Model saved successfully to './gpt-oss-20b-mytune-merged'


In [4]:
from safetensors import safe_open

for i in [1,2,3]:
    path = f"gpt-oss-20b-mytune-merged/model-0000{i}-of-00003.safetensors"
    try:
        with safe_open(path, framework="pt") as f:
            keys = list(f.keys())
        print(path, "OK -", len(keys), "tensors")
    except Exception as e:
        print(path, "LỖI:", e)

gpt-oss-20b-mytune-merged/model-00001-of-00003.safetensors OK - 4352 tensors
gpt-oss-20b-mytune-merged/model-00002-of-00003.safetensors OK - 5580 tensors
gpt-oss-20b-mytune-merged/model-00003-of-00003.safetensors OK - 1615 tensors


## Upload folder model lên huggingface

In [10]:
from huggingface_hub import login, create_repo, upload_folder

In [11]:
repo_id = "kazDE/gpt-oss-20b-mytune-merged"

In [12]:
create_repo(repo_id, repo_type="model", private=True)  # private=False nếu muốn public

# Upload cả thư mục
upload_folder(
    folder_path="./gpt-oss-20b-mytune-merged",
    repo_id=repo_id,
    repo_type="model",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...une-merged/tokenizer.json: 100%|##########| 27.9MB / 27.9MB            

  ...0003-of-00003.safetensors:   1%|1         | 29.4MB / 2.54GB            

  ...0002-of-00003.safetensors:   0%|          | 22.7MB / 5.00GB            

  ...0001-of-00003.safetensors:   0%|          | 22.7MB / 5.00GB            

CommitInfo(commit_url='https://huggingface.co/kazDE/gpt-oss-20b-mytune-merged/commit/bb872cfa6a74f96a681774d0ba6bb828e3864bcd', commit_message='Upload folder using huggingface_hub', commit_description='', oid='bb872cfa6a74f96a681774d0ba6bb828e3864bcd', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kazDE/gpt-oss-20b-mytune-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='kazDE/gpt-oss-20b-mytune-merged'), pr_revision=None, pr_num=None)